# Estimasi Usia Wajah di Google Colab
**Notebook mandiri:** seluruh kode audit, split, model, training, evaluasi,
grafik, dan inferensi berada di sel notebook ini. Tidak perlu clone GitHub,
mengunggah arsip source code, atau menyalin modul proyek.

## Cara mulai
1. Buat folder **My Drive / ComVis** dan unggah ZIP **UTKFace Aligned & Cropped**.
   Namai arsip `dataset.zip`; pertahankan nama asli gambar di dalamnya.
2. Buka notebook ini di Colab dan pilih **Runtime → Change runtime type → GPU**.
3. Jalankan sel dari atas ke bawah. Default menjalankan audit dan empat metode,
   lalu membandingkan hasil validasi. Evaluasi test dan unggah foto bersifat opsional.
4. Hasil tersimpan di **My Drive / ComVis / runs**. Notebook berisi kode,
   bukan checkpoint yang sudah dilatih; training perlu dijalankan dahulu.

Proyek memakai UTKFace, MobileNetV3 Small ImageNet, serta baseline B0, B1,
M1, dan B1 control. Label dataset tidak sama dengan kepastian umur kronologis.
Skor penelitian belum diisi. Logbook dibuat kosong untuk kamu isi sendiri.

## A. Pasang dependensi pendukung
PyTorch dan torchvision menggunakan instalasi bawaan runtime Colab.
Bobot ImageNet akan diunduh otomatis saat B1 dimulai. Sel ini memasang
pustaka pendukung langsung; tidak membaca requirements dari proyek.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'numpy>=1.26,<3', 'Pillow>=10,<13', 'matplotlib>=3.8,<4',
], check=True)
import torch
import torchvision
from IPython.display import display, Image as DisplayImage

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('CUDA tersedia:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Training akan memakai CPU. Pilih runtime GPU jika tersedia.')


## B. Sambungkan Drive dan tentukan dataset
Setujui dialog akses Drive. Ubah `DRIVE_PROJECT` atau `DATASET_ZIP` bila
lokasi/nama ZIP kamu berbeda. Folder dalam ZIP boleh dipertahankan;
pencarian gambar berjalan secara rekursif.

In [ ]:
import os
from pathlib import Path
from google.colab import drive
from IPython.display import display, Image as DisplayImage

drive.mount('/content/drive')
DRIVE_PROJECT = Path('/content/drive/MyDrive/ComVis')
DATASET_ZIP = DRIVE_PROJECT / 'dataset.zip'
RUNTIME_ROOT = Path('/content/ComVis_runtime')
if not DATASET_ZIP.is_file():
    raise FileNotFoundError(
        f'Dataset belum tersedia: {DATASET_ZIP}. '
        'Unggah ZIP UTKFace ke Drive atau sesuaikan path pada sel ini.'
    )
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('TORCH_HOME', str(RUNTIME_ROOT / '.cache' / 'torch'))
os.environ.setdefault('MPLCONFIGDIR', str(RUNTIME_ROOT / '.cache' / 'matplotlib'))
print('Arsip dataset:', DATASET_ZIP)
print('Hasil eksperimen:', DRIVE_PROJECT / 'runs')


## C. Fungsi penyimpanan dan metrik
JSON menyimpan konfigurasi dan hasil; CSV menyimpan manifest/prediksi.
Semua galat dihitung dari prediksi mentah dalam satuan tahun.

In [ ]:
import csv
import json
from pathlib import Path


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    temporary.replace(path)


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def write_csv(path, rows, fields):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)


def read_csv(path):
    with Path(path).open(newline='', encoding='utf-8-sig') as handle:
        return list(csv.DictReader(handle))


In [ ]:
import math
import statistics

AGE_GROUPS = ('0-12', '13-19', '20-39', '40-59', '60+')


def age_group(age):
    if not math.isfinite(float(age)) or not 0 <= float(age) <= 116:
        raise ValueError('Label usia harus berada pada rentang 0-116.')
    for boundary, label in [(12,'0-12'),(19,'13-19'),(39,'20-39'),(59,'40-59')]:
        if age <= boundary:
            return label
    return '60+'


def regression_metrics(targets, predictions):
    targets, predictions = list(targets), list(predictions)
    if not targets or len(targets) != len(predictions):
        raise ValueError('Label dan prediksi harus berjumlah sama dan tidak kosong.')
    if any(not math.isfinite(float(v)) for v in targets + predictions):
        raise ValueError('Label dan prediksi harus bernilai finite.')
    errors = [abs(float(a)-float(b)) for a,b in zip(targets,predictions)]
    return {'count':len(errors), 'mae':statistics.mean(errors),
            'rmse':math.sqrt(statistics.mean(e*e for e in errors)),
            'median_absolute_error':statistics.median(errors)}


def age_group_metrics(targets, predictions):
    targets, predictions = list(targets), list(predictions)
    regression_metrics(targets, predictions)
    result = {}
    for group in AGE_GROUPS:
        pairs = [(a,b) for a,b in zip(targets,predictions) if age_group(a)==group]
        result[group] = regression_metrics(*zip(*pairs)) if pairs else {
            'count':0, 'mae':None, 'rmse':None, 'median_absolute_error':None}
    return result


## D. Ekstrak dataset ke disk runtime
ZIP disalin dari Drive ke disk sementara, lalu hanya JPG/JPEG/PNG diekstrak.
Folder ekstraksi ditandai dengan hash isi arsip untuk mencegah campuran data.
Setelah runtime baru, jalankan sel ini lagi; dataset di Drive tetap tersimpan.

In [ ]:
"""Ekstraksi ZIP untuk bootstrap Colab dan dataset lokal."""
import shutil
import stat
import zipfile
from pathlib import Path, PurePosixPath


def extract_zip(archive, destination, images_only=False):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive) as stream:
        members = []
        seen = set()
        for entry in stream.infolist():
            name = entry.filename
            relative = PurePosixPath(name)
            if (relative.is_absolute() or '..' in relative.parts or '\\' in name
                    or ':' in name or stat.S_ISLNK(entry.external_attr >> 16)):
                raise ValueError(f'Path ZIP tidak aman: {name}')
            target = (destination / name).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(f'Path ZIP keluar folder tujuan: {name}')
            if entry.is_dir():
                continue
            if target in seen:
                raise ValueError(f'Path duplikat dalam ZIP: {name}')
            seen.add(target)
            if images_only and target.suffix.lower() not in {'.jpg', '.jpeg', '.png'}:
                continue
            members.append((entry, target))
        if not members:
            raise ValueError('Arsip tidak berisi file yang diperlukan.')
        for entry, target in members:
            target.parent.mkdir(parents=True, exist_ok=True)
            with stream.open(entry) as source, target.open('wb') as output:
                shutil.copyfileobj(source, output)
    return len(members)


In [ ]:
import hashlib
import shutil

def archive_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def stage_dataset(source, runtime_root):
    source, runtime_root = Path(source), Path(runtime_root)
    runtime_root.mkdir(parents=True, exist_ok=True)
    local_zip = runtime_root / 'dataset.zip'
    shutil.copy2(source, local_zip)
    digest = archive_hash(local_zip)
    destination = runtime_root / f'UTKFace_{digest[:12]}'
    marker = destination / '.archive_sha256'
    if marker.is_file():
        if marker.read_text(encoding='utf-8') != digest:
            raise RuntimeError('Hash folder berbeda; gunakan folder runtime baru.')
        print('Ekstraksi sebelumnya digunakan kembali.')
    else:
        count = extract_zip(local_zip, destination, images_only=True)
        marker.write_text(digest, encoding='utf-8')
        print(f'{count:,} citra diekstrak ke disk runtime.')
    return destination, digest

DATA, DATASET_SHA256 = stage_dataset(DATASET_ZIP, RUNTIME_ROOT)
print('Dataset:', DATA)
print('SHA256 arsip:', DATASET_SHA256)


## E. Fungsi audit dan pembagian data
Kode memeriksa label usia dari nama berkas, gambar rusak, hash file/piksel,
dan kemiripan dHash. Kelompok duplikat ditempatkan dalam satu split.
Split menargetkan 70% latih, 15% validasi, dan 15% test dengan seed 42.
Rasio aktual bergantung ukuran kelompok. dHash bukan verifikasi identitas.

In [ ]:
"""Audit UTKFace dan manifest split berbasis kelompok duplikat."""
import hashlib
import json
import random
import re
import statistics
from collections import Counter, defaultdict
from pathlib import Path, PurePosixPath

import numpy as np
from PIL import Image, ImageOps, UnidentifiedImageError


SPLITS = ('train','validation','test')
FIELDS = ['path','age','age_group','width','height','byte_sha256','pixel_sha256','dhash','group_id','split']


def parse_age(filename):
    match = re.match(r'^(\d+)_\d+_\d+_.+\.(?:jpg|jpeg|png)$', Path(filename).name, re.IGNORECASE)
    if not match or not 0 <= int(match[1]) <= 116:
        raise ValueError('Nama berkas UTKFace atau label usia tidak valid.')
    return int(match[1])


def pixel_hash(image):
    rgb = image.convert('RGB')
    return hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()


def dhash(image):
    values = np.asarray(image.convert('L').resize((9,8),Image.Resampling.LANCZOS))
    bits = (values[:,1:] > values[:,:-1]).ravel()
    result = 0
    for bit in bits:
        result = (result << 1) | int(bit)
    return result


class _UnionFind:
    def __init__(self, count): self.parent = list(range(count))

    def find(self, value):
        while self.parent[value] != value:
            self.parent[value] = self.parent[self.parent[value]]
            value = self.parent[value]
        return value

    def join(self, left, right):
        self.parent[self.find(right)] = self.find(left)


class _HashTree:
    """BK tree untuk pencarian Hamming tanpa membandingkan semua pasangan."""
    def __init__(self): self.root = None

    def nearby(self, value, radius):
        stack = [self.root] if self.root is not None else []
        while stack:
            node = stack.pop()
            distance = (value ^ node[0]).bit_count()
            if distance <= radius: yield node[1],distance
            stack.extend(child for delta,child in node[2].items()
                         if distance-radius <= delta <= distance+radius)

    def add(self, value, index):
        if self.root is None:
            self.root = [value,index,{}]
            return
        node = self.root
        while True:
            distance = (value ^ node[0]).bit_count()
            if distance == 0: return
            if distance not in node[2]:
                node[2][distance] = [value,index,{}]
                return
            node = node[2][distance]


def audit_images(root, near_threshold=4):
    root = Path(root).resolve()
    if not root.is_dir(): raise ValueError(f'Folder dataset tidak ditemukan: {root}')
    if near_threshold is not None and not 0 <= near_threshold <= 8:
        raise ValueError('Threshold kemiripan harus 0-8 atau dinonaktifkan.')
    rows, invalid, near = [], [], []
    paths = sorted(p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png'})
    for path in paths:
        name = path.relative_to(root).as_posix()
        try:
            if not path.resolve().is_relative_to(root): raise ValueError('Symlink keluar folder data.')
            age = parse_age(path.name)
            with Image.open(path) as original:
                original.load()
                image = ImageOps.exif_transpose(original).convert('RGB')
            rows.append({'path':name,'age':age,'age_group':age_group(age),
                         'width':image.width,'height':image.height,
                         'byte_sha256':hashlib.sha256(path.read_bytes()).hexdigest(),
                         'pixel_sha256':pixel_hash(image), 'dhash':f'{dhash(image):016x}',
                         '_texture':float(np.std(np.asarray(image.convert('L'))))})
        except (OSError, ValueError, UnidentifiedImageError, Image.DecompressionBombError) as exc:
            invalid.append({'path':name,'reason':str(exc)})
    if not rows: raise ValueError('Tidak ada citra UTKFace valid. Periksa folder dan nama berkas.')
    groups = _UnionFind(len(rows))
    exact, tree = {}, _HashTree()
    for index,row in enumerate(rows):
        pixel = row['pixel_sha256']
        if pixel in exact: groups.join(index,exact[pixel])
        else: exact[pixel] = index
        if near_threshold is not None and row['_texture'] >= 4:
            value = int(row['dhash'],16)
            for other,distance in tree.nearby(value,near_threshold):
                if rows[other]['pixel_sha256'] != pixel:
                    near.append({'left':rows[other]['path'],'right':row['path'],'hamming_distance':distance})
                groups.join(index,other)
            tree.add(value,index)
    keys = defaultdict(list)
    for index,row in enumerate(rows): keys[groups.find(index)].append(row['pixel_sha256'])
    for index,row in enumerate(rows):
        row['group_id'] = 'g_' + min(keys[groups.find(index)])[:20]
        row.pop('_texture')
    grouped = defaultdict(list)
    for row in rows: grouped[row['group_id']].append(row)
    report = {'source':'https://susanqq.github.io/UTKFace/', 'scanned_images':len(paths),
              'valid_images':len(rows), 'invalid_images':len(invalid),
              'unique_pixel_images':len(exact), 'duplicate_groups':sum(len(v)>1 for v in grouped.values()),
              'independent_groups':len(grouped), 'near_pairs':len(near),
              'near_threshold':near_threshold,
              'conflicting_age_groups':sum(len({r['age'] for r in v})>1 for v in grouped.values()),
              'age_distribution':dict(sorted(Counter(r['age'] for r in rows).items())),
              'age_group_distribution':dict(Counter(r['age_group'] for r in rows)),
              'limitations':['Labels berasal dari estimasi DEX dan pemeriksaan manusia.',
                            'dHash mengelompokkan kemiripan konservatif; bukan verifikasi identitas.',
                            'Pemisahan berdasarkan orang yang sama tidak dapat dijamin.']}
    return rows, report, invalid, near


def assign_splits(rows, seed=42, ratios=(0.7,0.15,0.15)):
    if len(ratios)!=3 or any(r<=0 for r in ratios) or abs(sum(ratios)-1)>1e-8:
        raise ValueError('Rasio split harus tiga nilai positif dengan total 1.')
    rows = [dict(row) for row in rows]
    grouped = defaultdict(list)
    for row in sorted(rows,key=lambda r:r['path']): grouped[row['group_id']].append(row)
    if len(grouped)<3: raise ValueError('Minimal tiga kelompok independen diperlukan untuk train/validation/test.')
    buckets = defaultdict(list)
    for key, members in sorted(grouped.items()):
        buckets[age_group(statistics.median(int(r['age']) for r in members))].append((key,members))
    rng, assigned, totals = random.Random(seed), {}, Counter()
    for bucket in AGE_GROUPS:
        items = buckets[bucket]
        rng.shuffle(items)
        items.sort(key=lambda x:len(x[1]),reverse=True)
        counts = Counter()
        for key,members in items:
            split_index = min(range(3),key=lambda i:(counts[SPLITS[i]]/ratios[i],
                                                   totals[SPLITS[i]]/ratios[i],i))
            split = SPLITS[split_index]
            assigned[key] = split
            counts[split] += len(members)
            totals[split] += len(members)
    # Sparse age buckets may each contain one group; keep all three sets usable.
    for missing in SPLITS:
        if totals[missing] == 0:
            candidates = [key for key,members in grouped.items()
                          if sum(v==assigned[key] for v in assigned.values())>1]
            key = min(candidates,key=lambda k:(len(grouped[k]),k))
            totals[assigned[key]] -= len(grouped[key]); assigned[key] = missing
            totals[missing] += len(grouped[key])
    for row in rows: row['split'] = assigned[row['group_id']]
    rows.sort(key=lambda r:r['path'])
    validate_manifest(rows)
    return rows


def validate_manifest(rows):
    if not rows: raise ValueError('Manifest kosong.')
    seen, groups, pixels = set(), {}, {}
    for row in rows:
        name = row['path'].replace('\\','/')
        path = PurePosixPath(name)
        if path.is_absolute() or '..' in path.parts or ':' in name or not name:
            raise ValueError('Path manifest harus relatif dan tetap di dalam folder data.')
        if name in seen: raise ValueError('Path citra ganda dalam manifest.')
        seen.add(name)
        age = int(row['age'])
        if str(age)!=str(row['age']) or not 0<=age<=116: raise ValueError('Usia manifest tidak valid.')
        split = row['split']
        if split not in SPLITS: raise ValueError('Nama split tidak dikenal.')
        for container,key in [(groups,row['group_id']), (pixels,row.get('pixel_sha256'))]:
            if key:
                if key in container and container[key]!=split:
                    raise ValueError('Kebocoran kelompok duplikat melintasi split.')
                container[key] = split


def load_manifest(path):
    rows = read_csv(path)
    for row in rows: row['age'] = int(row['age'])
    validate_manifest(rows)
    return rows


def manifest_fingerprint(rows):
    validate_manifest(rows)
    canonical = [{k:r[k] for k in ['path','age','group_id','split','pixel_sha256']} for r in sorted(rows,key=lambda r:r['path'])]
    return hashlib.sha256(json.dumps(canonical,sort_keys=True).encode()).hexdigest()


def prepare_data(root, output, seed=42, near_threshold=4, force=False):
    output = Path(output)
    if (output/'manifest.csv').exists() and not force:
        raise ValueError('Manifest sudah ada; gunakan split ini atau --force untuk audit baru sebelum eksperimen.')
    rows,report,invalid,near = audit_images(root,near_threshold)
    manifest = assign_splits(rows,seed)
    report.update({'seed':seed, 'requested_ratios':dict(zip(SPLITS,[0.7,0.15,0.15])),
                   'split_counts':dict(Counter(r['split'] for r in manifest)),
                   'fingerprint':manifest_fingerprint(manifest),
                   'split_age_groups':{s:dict(Counter(age_group(r['age']) for r in manifest if r['split']==s)) for s in SPLITS}})
    write_csv(output/'inventory.csv',rows,FIELDS[:-1])
    write_csv(output/'manifest.csv',manifest,FIELDS)
    write_csv(output/'invalid_files.csv',invalid,['path','reason'])
    write_csv(output/'near_duplicates.csv',near,['left','right','hamming_distance'])
    count = Counter(r['group_id'] for r in manifest)
    write_csv(output/'duplicate_groups.csv',[r for r in manifest if count[r['group_id']]>1],FIELDS)
    write_json(output/'audit.json',report)
    return report


def verify_data_files(root, rows):
    root = Path(root).resolve()
    for row in rows:
        path = (root/row['path']).resolve()
        if not path.is_relative_to(root): raise ValueError('Path citra keluar folder data.')
        if not path.is_file(): raise ValueError(f'Citra tidak ditemukan: {row["path"]}')
        if row.get('byte_sha256') and hashlib.sha256(path.read_bytes()).hexdigest()!=row['byte_sha256']:
            raise ValueError(f'Citra berubah setelah audit: {row["path"]}. Jalankan audit baru sebelum eksperimen.')


## F. Model MobileNetV3 Small dan dataset PyTorch
Citra RGB di-resize 256 pada sisi pendek, di-center crop 224×224, lalu
dinormalisasi dengan statistik ImageNet. Augmentasi hanya untuk data latih.
Classifier menghasilkan satu angka usia. B1/B1 control membekukan backbone
dan BatchNorm; M1 membuka tiga modul terakhir backbone beserta head.

In [ ]:
import torch
from torch import nn
from torchvision import models, transforms

MEAN = [0.485,0.456,0.406]
STD = [0.229,0.224,0.225]
PREPROCESS = {'resize':256, 'crop':224, 'mean':MEAN, 'std':STD}


def make_transform(training=False, augmentation=True):
    operations = [transforms.Resize(256), transforms.CenterCrop(224)]
    if training and augmentation:
        operations += [transforms.RandomHorizontalFlip(),
                       transforms.ColorJitter(brightness=0.1,contrast=0.1)]
    return transforms.Compose(operations + [transforms.ToTensor(),transforms.Normalize(MEAN,STD)])


def create_model(weights='IMAGENET1K_V1', train_median=0.0):
    if weights not in {'IMAGENET1K_V1','none'}: raise ValueError('Bobot pralatih tidak dikenal.')
    source = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1 if weights!='none' else None
    model = models.mobilenet_v3_small(weights=source)
    previous = model.classifier[-1]
    model.classifier[-1] = nn.Linear(previous.in_features,1)
    nn.init.constant_(model.classifier[-1].bias,float(train_median))
    return model


def configure_training(model, method, unfreeze_blocks=3):
    if method not in {'b1','m1','b1_control'}: raise ValueError('Metode jaringan tidak dikenal.')
    if not isinstance(unfreeze_blocks,int) or not 1<=unfreeze_blocks<=len(model.features):
        raise ValueError('Jumlah modul fine tuning harus berada dalam rentang backbone.')
    for parameter in model.parameters(): parameter.requires_grad = False
    for parameter in model.classifier.parameters(): parameter.requires_grad = True
    if method=='m1':
        for layer in model.features[-unfreeze_blocks:]:
            for parameter in layer.parameters(): parameter.requires_grad = True


def set_training_mode(model, method, unfreeze_blocks=3):
    model.train()
    model.features.eval()
    if method=='m1':
        for layer in model.features[-unfreeze_blocks:]: layer.train()


def resolve_device(request='auto'):
    if request not in {'auto','cpu','cuda'}: raise ValueError('Device harus auto, cpu, atau cuda.')
    if request=='cuda' and not torch.cuda.is_available(): raise ValueError('GPU CUDA tidak tersedia.')
    return torch.device('cuda' if request=='cuda' or (request=='auto' and torch.cuda.is_available()) else 'cpu')


In [ ]:
from pathlib import Path

import torch
from PIL import Image, ImageOps
from torch.utils.data import Dataset


class FaceDataset(Dataset):
    def __init__(self, root, rows, transform):
        self.root = Path(root).resolve()
        self.rows = list(rows)
        self.transform = transform

    def __len__(self): return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        path = (self.root/row['path']).resolve()
        if not path.is_relative_to(self.root): raise ValueError('Citra keluar dari folder data.')
        with Image.open(path) as source:
            source.load()
            image = ImageOps.exif_transpose(source).convert('RGB')
        return self.transform(image),torch.tensor(float(row['age']),dtype=torch.float32),row['path']


## G. Fungsi training
Sel ini mendefinisikan training; belum menjalankan eksperimen.
Optimizer AdamW, loss Smooth L1, checkpoint terbaik menurut MAE validasi,
serta early stopping digunakan oleh seluruh metode jaringan.
M1 dan B1 control dimulai dari checkpoint B1 yang sama.
Catatan runtime mencantumkan perangkat dan versi pustaka yang benar-benar dipakai.

In [ ]:
import hashlib
import math
import os
import platform
import random
import statistics
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torchvision
from torch.utils.data import DataLoader, WeightedRandomSampler


DEFAULT_CONFIG = {'seed':42,'epochs':30,'batch_size':32,'patience':5,
                  'learning_rate':0.0001,'backbone_learning_rate':0.00001,'weight_decay':0.0001,
                  'loss_beta':1.0,'unfreeze_blocks':3,'augmentation':True,'sampling':'natural',
                  'weights':'IMAGENET1K_V1','num_workers':0,'cpu_threads':4,'dataset_kind':'utkface'}


def validate_config(overrides=None):
    overrides = overrides or {}
    unknown = set(overrides)-set(DEFAULT_CONFIG)
    if unknown: raise ValueError(f'Konfigurasi tidak dikenal: {sorted(unknown)}')
    config = dict(DEFAULT_CONFIG,**overrides)
    for key in ['epochs','batch_size','patience','unfreeze_blocks','cpu_threads']:
        if type(config[key]) is not int or config[key]<1: raise ValueError(f'{key} harus integer positif.')
    if type(config['num_workers']) is not int or config['num_workers']<0: raise ValueError('num_workers tidak valid.')
    if type(config['seed']) is not int or not 0<=config['seed']<2**32: raise ValueError('seed tidak valid.')
    for key in ['learning_rate','backbone_learning_rate','loss_beta','weight_decay']:
        if not isinstance(config[key],(int,float)) or not math.isfinite(config[key]) or config[key]<0:
            raise ValueError(f'{key} harus angka finite nonnegatif.')
        if key!='weight_decay' and config[key]==0: raise ValueError(f'{key} harus positif.')
    if config['sampling'] not in {'natural','balanced'}: raise ValueError('Sampling tidak dikenal.')
    if config['weights'] not in {'IMAGENET1K_V1','none'}: raise ValueError('Bobot tidak dikenal.')
    if type(config['augmentation']) is not bool: raise ValueError('augmentation harus boolean.')
    if config['dataset_kind'] not in {'utkface','synthetic'}: raise ValueError('dataset_kind tidak dikenal.')
    if config['dataset_kind']=='utkface' and config['weights']=='none':
        raise ValueError('Eksperimen proposal menggunakan bobot ImageNet; bobot acak hanya untuk smoke sintetis.')
    return config


def seed_everything(seed, threads=4):
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.set_num_threads(threads)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True,warn_only=True)


def _worker_seed(_):
    seed = torch.initial_seed() % 2**32
    np.random.seed(seed); random.seed(seed)


def environment_info(device):
    return {'python':platform.python_version(),'platform':platform.platform(),
            'torch':str(torch.__version__),'torchvision':str(torchvision.__version__),
            'numpy':str(np.__version__),'device':str(device),
            'gpu_name':torch.cuda.get_device_name(device) if device.type=='cuda' else None,
            'gpu_memory_bytes':torch.cuda.get_device_properties(device).total_memory if device.type=='cuda' else None,
            'cuda':torch.version.cuda, 'cpu':platform.processor(),
            'cpu_threads':torch.get_num_threads()}


def predict_loader(model, loader, device):
    model.eval()
    predictions, targets, paths = [], [], []
    with torch.inference_mode():
        for images, labels, names in loader:
            output = model(images.to(device)).flatten().cpu().tolist()
            predictions.extend(output); targets.extend(labels.tolist()); paths.extend(names)
    return targets,predictions,paths


def load_checkpoint(path, device='cpu'):
    payload = torch.load(path,map_location=device,weights_only=True)
    if not isinstance(payload,dict) or payload.get('schema_version')!=1:
        raise ValueError('Format checkpoint tidak didukung.')
    if payload.get('kind')!='mobilenet_v3_small_age_regression' or 'state_dict' not in payload:
        raise ValueError('Checkpoint bukan model estimasi usia proyek ini.')
    metadata = payload.get('metadata',{})
    if metadata.get('preprocess') != PREPROCESS or not metadata.get('manifest_fingerprint'):
        raise ValueError('Metadata praproses atau split checkpoint tidak valid.')
    if payload.get('epoch',0)<1: raise ValueError('Checkpoint belum menjalankan pelatihan.')
    model = create_model(weights='none')
    model.load_state_dict(payload['state_dict'],strict=True)
    model.to(device).eval()
    return model, metadata, payload


def train(method, root, manifest_path, output, config=None, device='auto', init_checkpoint=None):
    if method not in {'b0','b1','m1','b1_control'}: raise ValueError('Metode tidak dikenal.')
    config = validate_config(config)
    rows = load_manifest(manifest_path)
    fingerprint = manifest_fingerprint(rows)
    split_rows = {s:[r for r in rows if r['split']==s] for s in SPLITS}
    if any(not value for value in split_rows.values()): raise ValueError('Semua split harus memiliki citra.')
    verify_data_files(root,split_rows['train']+split_rows['validation'])
    seed_everything(config['seed'],config['cpu_threads'])
    device = resolve_device(device)
    median = statistics.median(row['age'] for row in split_rows['train'])
    if method in {'m1','b1_control'} and not init_checkpoint:
        raise ValueError('M1 dan B1 control memerlukan --init-checkpoint dari B1.')
    if method in {'b0','b1'} and init_checkpoint:
        raise ValueError('Checkpoint awal hanya untuk M1 atau B1 control.')
    source_hash = None
    model = None
    if init_checkpoint:
        model,source,_ = load_checkpoint(init_checkpoint,device)
        if source.get('method')!='b1' or source['manifest_fingerprint']!=fingerprint:
            raise ValueError('Checkpoint awal harus B1 dengan manifest yang sama.')
        for key in ['seed','batch_size','augmentation','sampling','weights','dataset_kind']:
            if source['config'][key]!=config[key]: raise ValueError(f'{key} berbeda dari checkpoint B1.')
        source_hash = hashlib.sha256(Path(init_checkpoint).read_bytes()).hexdigest()
    elif method!='b0':
        model = create_model(config['weights'],median).to(device)
    output = Path(output)
    if output.exists() and any(output.iterdir()): raise ValueError('Folder run sudah berisi hasil; pilih nama run baru.')
    output.mkdir(parents=True,exist_ok=True)
    metadata = {'schema_version':1,'method':method,'config':config,'preprocess':PREPROCESS,
                'manifest_fingerprint':fingerprint,'source_checkpoint_sha256':source_hash,
                'data_counts':{s:len(v) for s,v in split_rows.items()},
                'created_utc':datetime.now(timezone.utc).isoformat(),
                'environment':environment_info(device),'train_median':median,
                'status':'training','research_model':config['dataset_kind']=='utkface' and config['weights']!='none'}
    write_json(output/'run.json',metadata); write_json(output/'config.json',config)
    start = time.perf_counter()
    if method=='b0':
        targets=[r['age'] for r in split_rows['validation']]
        metrics=regression_metrics(targets,[median]*len(targets))
        metadata.update({'status':'complete','training_seconds':time.perf_counter()-start,
                         'best_epoch':0,'best_validation_mae':metrics['mae'],'parameter_count':0})
        write_json(output/'b0.json',{'kind':'train_median_baseline','median':median,'metadata':metadata})
        write_csv(output/'history.csv',[{'epoch':0,'train_mae':None,'validation_mae':metrics['mae']}],
                  ['epoch','train_mae','validation_mae'])
    else:
        configure_training(model,method,config['unfreeze_blocks'])
        train_dataset=FaceDataset(root,split_rows['train'],make_transform(True,config['augmentation']))
        validation_dataset=FaceDataset(root,split_rows['validation'],make_transform(False))
        generator=torch.Generator().manual_seed(config['seed'])
        sampler=None
        if config['sampling']=='balanced':
            counts=Counter(age_group(r['age']) for r in split_rows['train'])
            weights=[1/counts[age_group(r['age'])] for r in split_rows['train']]
            sampler=WeightedRandomSampler(weights,len(weights),replacement=True,generator=generator)
        train_loader=DataLoader(train_dataset,batch_size=config['batch_size'],shuffle=sampler is None,
                                sampler=sampler,num_workers=config['num_workers'],worker_init_fn=_worker_seed,
                                generator=generator,pin_memory=device.type=='cuda')
        validation_loader=DataLoader(validation_dataset,batch_size=config['batch_size'],shuffle=False,
                                     num_workers=config['num_workers'])
        parameters=[{'params':list(model.classifier.parameters()),'lr':config['learning_rate']}]
        backbone=[p for p in model.features.parameters() if p.requires_grad]
        if backbone: parameters.append({'params':backbone,'lr':config['backbone_learning_rate']})
        optimizer=torch.optim.AdamW(parameters,weight_decay=config['weight_decay'])
        loss_fn=torch.nn.SmoothL1Loss(beta=config['loss_beta'])
        history,best,stale,best_epoch=[],float('inf'),0,0
        for epoch in range(1,config['epochs']+1):
            set_training_mode(model,method,config['unfreeze_blocks'])
            loss_total,error_total,count=0.0,0.0,0
            for images, labels, _ in train_loader:
                images,labels=images.to(device),labels.to(device)
                optimizer.zero_grad(set_to_none=True)
                predicted=model(images).flatten()
                loss=loss_fn(predicted,labels)
                if not torch.isfinite(loss): raise ValueError('Loss tidak finite; periksa konfigurasi dan data.')
                loss.backward(); optimizer.step()
                loss_total+=float(loss.detach())*len(labels)
                error_total+=float((predicted.detach()-labels).abs().sum())
                count+=len(labels)
            targets,predictions,_=predict_loader(model,validation_loader,device)
            val=regression_metrics(targets,predictions)
            row={'epoch':epoch,'train_loss':loss_total/count,'train_mae':error_total/count,
                 'validation_mae':val['mae'],'validation_rmse':val['rmse'],
                 'elapsed_seconds':time.perf_counter()-start}
            history.append(row)
            write_csv(output/'history.csv',history,list(row))
            print(f'{method} epoch {epoch}/{config["epochs"]} train MAE={row["train_mae"]:.3f} val MAE={val["mae"]:.3f}',flush=True)
            if val['mae']<best:
                best,best_epoch,stale=val['mae'],epoch,0
                snapshot={'schema_version':1,'kind':'mobilenet_v3_small_age_regression',
                          'state_dict':{k:v.detach().cpu() for k,v in model.state_dict().items()},
                          'epoch':epoch,'validation_mae':best,'metadata':dict(metadata)}
                temporary=output/'best.pt.tmp'; torch.save(snapshot,temporary); temporary.replace(output/'best.pt')
            else: stale+=1
            if stale>=config['patience']: break
        metadata.update({'status':'complete','best_epoch':best_epoch,'best_validation_mae':best,
                         'epochs_completed':len(history),'training_seconds':time.perf_counter()-start,
                         'parameter_count':sum(p.numel() for p in model.parameters()),
                         'trainable_parameter_count':sum(p.numel() for p in model.parameters() if p.requires_grad)})
    write_json(output/'run.json',metadata)
    evaluate(output,root,manifest_path,'validation',device=str(device))
    return metadata


## H. Fungsi evaluasi, grafik, dan inferensi
MAE, RMSE, median absolute error, dan galat per kelompok usia dilaporkan.
Prediksi, kasus terbaik/terburuk, dan grafik disimpan ke Drive.
Test set dikunci sampai `RUN_FINAL_TEST` diaktifkan.
Foto inferensi harus berisi satu wajah yang sudah dipotong; tidak ada deteksi otomatis.

In [ ]:
import math
import os
import time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader



def _pyplot():
    os.environ.setdefault('MPLCONFIGDIR',str(Path('.cache/matplotlib').resolve()))
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    return plt


def create_plots(output, targets, predictions, history_path=None):
    plt=_pyplot()
    output=Path(output)
    targets,predictions=np.asarray(targets),np.asarray(predictions)
    fig,ax=plt.subplots(figsize=(6,5))
    ax.scatter(targets,predictions,s=12,alpha=.35,color='#2563eb')
    low=float(min(targets.min(),predictions.min(),0)); high=float(max(targets.max(),predictions.max(),116))
    ax.plot([low,high],[low,high],'--',color='#475569')
    ax.set(xlabel='Label usia (tahun)',ylabel='Prediksi mentah (tahun)',title='Prediksi terhadap label usia')
    fig.tight_layout(); fig.savefig(output/'prediction_vs_label.png',dpi=170); plt.close(fig)
    fig,ax=plt.subplots(figsize=(6,4))
    ax.hist(predictions-targets,bins=min(30,max(3,len(targets))),color='#2563eb',edgecolor='white')
    ax.axvline(0,color='#475569',linestyle='--')
    ax.set(xlabel='Residual prediksi dikurangi label (tahun)',ylabel='Jumlah citra',title='Distribusi residual')
    fig.tight_layout(); fig.savefig(output/'residual_histogram.png',dpi=170); plt.close(fig)
    groups=age_group_metrics(targets.tolist(),predictions.tolist())
    fig,ax=plt.subplots(figsize=(6,4))
    ax.bar(list(groups),[v['mae'] if v['mae'] is not None else 0 for v in groups.values()],color='#2563eb')
    ax.set(xlabel='Kelompok label usia',ylabel='MAE (tahun)',title='Galat per kelompok usia')
    for i,v in enumerate(groups.values()): ax.text(i,0,f'n={v["count"]}',ha='center',va='bottom',fontsize=8)
    fig.tight_layout(); fig.savefig(output/'age_group_mae.png',dpi=170); plt.close(fig)
    if history_path and Path(history_path).is_file():
        rows=read_csv(history_path)
        if rows and rows[0].get('train_mae') not in {'',None}:
            fig,ax=plt.subplots(figsize=(6,4))
            ax.plot([int(r['epoch']) for r in rows],[float(r['train_mae']) for r in rows],label='Latih')
            ax.plot([int(r['epoch']) for r in rows],[float(r['validation_mae']) for r in rows],label='Validasi')
            ax.set(xlabel='Epoch',ylabel='MAE (tahun)',title='Kurva pelatihan'); ax.legend()
            fig.tight_layout(); fig.savefig(output/'learning_curve.png',dpi=170); plt.close(fig)


def evaluate(run, root, manifest_path, split='validation', device='auto', final_test=False):
    if split not in {'validation','test'}: raise ValueError('Evaluasi hanya validation atau test.')
    if split=='test' and not final_test:
        raise ValueError('Test set dikunci untuk evaluasi akhir; tambahkan --final-test setelah memilih konfigurasi.')
    run=Path(run)
    metadata=read_json(run/'run.json')
    if metadata.get('status')!='complete': raise ValueError('Run belum selesai.')
    rows=load_manifest(manifest_path)
    fingerprint=manifest_fingerprint(rows)
    if fingerprint!=metadata['manifest_fingerprint']:
        raise ValueError('Manifest berbeda dari pelatihan; perbandingan tidak valid.')
    selected=[r for r in rows if r['split']==split]
    if not selected: raise ValueError('Split evaluasi kosong.')
    verify_data_files(root,selected)
    device=resolve_device(device)
    torch.set_num_threads(metadata['config']['cpu_threads'])
    start=time.perf_counter()
    if metadata['method']=='b0':
        payload=read_json(run/'b0.json')
        targets=[r['age'] for r in selected]; predictions=[payload['median']]*len(selected)
        paths=[r['path'] for r in selected]
    else:
        model,checkpoint_metadata,_=load_checkpoint(run/'best.pt',device)
        if checkpoint_metadata['manifest_fingerprint']!=fingerprint: raise ValueError('Checkpoint dan manifest berbeda.')
        if checkpoint_metadata.get('method')!=metadata['method'] or checkpoint_metadata.get('config')!=metadata['config']:
            raise ValueError('Checkpoint tertukar atau konfigurasi run berbeda dari model yang dimuat.')
        loader=DataLoader(FaceDataset(root,selected,make_transform(False)),
                          batch_size=metadata['config']['batch_size'],shuffle=False,
                          num_workers=metadata['config']['num_workers'])
        targets,predictions,paths=predict_loader(model,loader,device)
    elapsed=time.perf_counter()-start
    scores=regression_metrics(targets,predictions)
    output=run/f'evaluation_{split}'; output.mkdir(exist_ok=True)
    prediction_rows=[{'path':path,'age':age,'age_group':age_group(age),'prediction':pred,
                      'residual':pred-age,'absolute_error':abs(pred-age)}
                     for path,age,pred in zip(paths,targets,predictions)]
    write_csv(output/'predictions.csv',prediction_rows,list(prediction_rows[0]))
    ordered=sorted(prediction_rows,key=lambda r:r['absolute_error'])
    write_csv(output/'best_examples.csv',ordered[:10],list(prediction_rows[0]))
    write_csv(output/'worst_examples.csv',list(reversed(ordered[-10:])),list(prediction_rows[0]))
    per_group=age_group_metrics(targets,predictions)
    write_csv(output/'age_groups.csv',[dict(age_group=group,**value) for group,value in per_group.items()],
              ['age_group','count','mae','rmse','median_absolute_error'])
    report={'method':metadata['method'],'split':split,'manifest_fingerprint':fingerprint,
            'dataset_kind':metadata['config']['dataset_kind'],'research_model':metadata['research_model'],
            'metrics':scores,'age_groups':per_group,'evaluation_seconds':elapsed,
            'amortized_ms_per_image_including_loading':elapsed*1000/len(targets),
            'device':str(device),'raw_predictions':True,'created_from_checkpoint':'best.pt' if metadata['method']!='b0' else 'b0.json'}
    write_json(output/'metrics.json',report)
    create_plots(output,targets,predictions,run/'history.csv')
    return report


def compare_runs(runs, root, manifest_path, output, split='validation', device='auto', final_test=False):
    runs=[Path(run) for run in runs]
    if len(runs)<2: raise ValueError('Perbandingan memerlukan minimal dua run.')
    metadata=[read_json(run/'run.json') for run in runs]
    fingerprint=manifest_fingerprint(load_manifest(manifest_path))
    if any(value.get('manifest_fingerprint')!=fingerprint for value in metadata):
        raise ValueError('Semua run harus menggunakan manifest yang sama.')
    if len({value['config']['dataset_kind'] for value in metadata})!=1:
        raise ValueError('Run sintetis dan penelitian tidak boleh dibandingkan.')
    if len(set(str(run.resolve()) for run in runs))!=len(runs): raise ValueError('Run yang sama dicantumkan dua kali.')
    rows=[]
    for run,meta in zip(runs,metadata):
        report=evaluate(run,root,manifest_path,split,device,final_test)
        rows.append({'run':run.name,'method':meta['method'],**report['metrics'],
                     'best_epoch':meta['best_epoch'],'training_seconds':meta['training_seconds'],
                     'parameter_count':meta['parameter_count']})
    baseline=next((r for r in rows if r['method']=='b0'),None)
    main=next((r for r in rows if r['method']=='m1'),None)
    improvement=None
    if baseline and main and baseline['mae']>0: improvement=(baseline['mae']-main['mae'])/baseline['mae']*100
    m1=next((m for m in metadata if m['method']=='m1'),None)
    control=next((m for m in metadata if m['method']=='b1_control'),None)
    same_source=bool(m1 and control and m1['source_checkpoint_sha256'] and
                     m1['source_checkpoint_sha256']==control['source_checkpoint_sha256'])
    controlled_keys=['epochs','batch_size','patience','learning_rate','backbone_learning_rate',
                     'weight_decay','loss_beta','seed','augmentation','sampling','dataset_kind','weights']
    same_settings=bool(m1 and control and all(m1['config'][key]==control['config'][key] for key in controlled_keys))
    summary={'split':split,'manifest_fingerprint':fingerprint,'rows':rows,
             'dataset_kind':metadata[0]['config']['dataset_kind'],
             'm1_mae_improvement_percent_over_b0':improvement,
             'target_20_percent_met':improvement>=20 if improvement is not None else None,
             'controlled_finetuning':{'same_initial_checkpoint':same_source,'same_training_settings':same_settings,
                                      'valid_control':same_source and same_settings,
                                      'note':'B1 control dan M1 memiliki batas epoch sama; early stopping dapat memberi jumlah epoch aktual berbeda.'}}
    output=Path(output); output.mkdir(parents=True,exist_ok=True)
    write_csv(output/f'comparison_{split}.csv',rows,list(rows[0]))
    write_json(output/f'comparison_{split}.json',summary)
    plt=_pyplot(); fig,ax=plt.subplots(figsize=(7,4))
    ax.bar([r['run'] for r in rows],[r['mae'] for r in rows],color='#2563eb')
    ax.set(ylabel='MAE (tahun)',title=f'Perbandingan pada split {split}'); ax.tick_params(axis='x',labelrotation=15)
    fig.tight_layout(); fig.savefig(output/f'comparison_{split}.png',dpi=170); plt.close(fig)
    return summary


In [ ]:
import io
import math
import time
from pathlib import Path

import torch
from PIL import Image, ImageOps



def decode_image(content):
    if not content or len(content)>10*1024*1024:
        raise ValueError('Unggah JPG atau PNG berukuran paling besar 10 MB.')
    with Image.open(io.BytesIO(content)) as original:
        if original.format not in {'JPEG','PNG'}: raise ValueError('Format gambar harus JPG atau PNG.')
        original.load()
        return ImageOps.exif_transpose(original).convert('RGB')


class Predictor:
    def __init__(self, checkpoint, device='cpu', allow_synthetic=False):
        checkpoint = Path(checkpoint)
        if not checkpoint.is_file(): raise ValueError('Checkpoint tidak ditemukan. Jalankan pelatihan dahulu.')
        self.device = resolve_device(device)
        self.model, self.median = None, None
        self.transform = make_transform(False)
        if checkpoint.suffix.lower()=='.json':
            payload=read_json(checkpoint)
            if payload.get('kind')!='train_median_baseline': raise ValueError('Berkas bukan baseline usia.')
            self.median=float(payload['median'])
            if not math.isfinite(self.median) or not 0<=self.median<=116: raise ValueError('Median baseline tidak valid.')
            self.metadata=payload['metadata']
            self.epoch=0
        else:
            self.model,self.metadata,payload=load_checkpoint(checkpoint,self.device)
            self.epoch=payload['epoch']
        if not self.metadata.get('research_model',False) and not allow_synthetic:
            raise ValueError('Checkpoint sintetis hanya untuk uji teknis, bukan estimasi usia wajah.')
        if self.model is not None:
            with torch.inference_mode(): self.model(torch.zeros(1,3,224,224,device=self.device))
            if self.device.type=='cuda': torch.cuda.synchronize(self.device)

    def predict(self, image):
        if not isinstance(image,Image.Image): raise ValueError('Masukan harus berupa gambar PIL.')
        start=time.perf_counter()
        if self.model is None: predicted=self.median
        else:
            tensor=self.transform(ImageOps.exif_transpose(image).convert('RGB')).unsqueeze(0).to(self.device)
            with torch.inference_mode(): predicted=float(self.model(tensor).flatten()[0].cpu())
            if self.device.type=='cuda': torch.cuda.synchronize(self.device)
        if not math.isfinite(predicted): raise ValueError('Prediksi tidak finite; periksa model.')
        return {'raw_age':predicted,'display_age':round(min(116,max(0,predicted)),1),
                'latency_ms':(time.perf_counter()-start)*1000,
                'method':self.metadata['method'],'device':str(self.device),
                'research_model':self.metadata.get('research_model',False),
                'note':'Keluaran model terhadap label dataset; bukan penentuan umur resmi.'}


def available_models(runs_root):
    root=Path(runs_root)
    result=[]
    if not root.is_dir(): return result
    for metadata_path in sorted(root.glob('*/run.json')):
        try:
            metadata=read_json(metadata_path)
            if not isinstance(metadata,dict): continue
            if not metadata.get('research_model') or metadata.get('status')!='complete': continue
            if metadata.get('method') not in {'b0','b1','m1','b1_control'}: continue
            score=float(metadata['best_validation_mae'])
            if not math.isfinite(score) or score<0 or int(metadata['data_counts']['train'])<1: continue
            filename='b0.json' if metadata['method']=='b0' else 'best.pt'
            checkpoint=metadata_path.parent/filename
            if checkpoint.is_file(): result.append({'path':checkpoint,'metadata':metadata,'name':metadata_path.parent.name})
        except (ValueError,TypeError,KeyError,OSError): continue
    return result


## 1. Konfigurasi eksperimen
Default langsung menjalankan audit, training, dan perbandingan validasi.
Untuk uji jalan singkat pada UTKFace, ubah `CONFIG['epochs']` menjadi 2
sebelum training dan gunakan `RUN_PREFIX` khusus, misalnya `uji02`.
Untuk eksperimen penuh gunakan nama baru, misalnya `colab02`, dan 30 epoch.

Folder run lama tidak ditimpa. Tahap yang telah selesai dengan konfigurasi
dan split sama digunakan kembali. Jika tahap terputus, gunakan nama run baru
untuk mengulang; belum ada resume optimizer/epoch. Jika mengganti ZIP,
seed, atau threshold kemiripan, ganti `DATA_VERSION` dan `RUN_PREFIX`.

In [ ]:
DATA_VERSION = 'utkface_v1'
RUN_PREFIX = 'colab01'
DEVICE = 'auto'  # auto / cpu / cuda
NEAR_THRESHOLD = 4  # 0–8; None hanya bila audit kemiripan sengaja dinonaktifkan
CONFIG = validate_config({
    'seed': 42,
    'epochs': 30,
    'batch_size': 32,
    'patience': 5,
    'learning_rate': 0.0001,
    'backbone_learning_rate': 0.00001,
    'weight_decay': 0.0001,
    'loss_beta': 1.0,
    'unfreeze_blocks': 3,
    'augmentation': True,
    'sampling': 'natural',
    'weights': 'IMAGENET1K_V1',
    'num_workers': 0,
    'cpu_threads': 4,
    'dataset_kind': 'utkface',
})

RUN_PREPARE = True
RUN_TRAINING = True
RUN_COMPARE = True
RUN_FINAL_TEST = False
SELECTED_METHOD = 'm1'  # pilihan awal; putuskan dari skor VALIDASI

PROCESSED = DRIVE_PROJECT / 'processed' / DATA_VERSION
MANIFEST = PROCESSED / 'manifest.csv'
RUNS = {method: DRIVE_PROJECT / 'runs' / f'{RUN_PREFIX}_{method}'
        for method in ('b0', 'b1', 'm1', 'b1_control')}
COMPARISON = DRIVE_PROJECT / 'runs' / f'{RUN_PREFIX}_comparison'
print(json.dumps(CONFIG, indent=2))


In [ ]:
def prepare_notebook_data():
    audit_path = PROCESSED / 'audit.json'
    if MANIFEST.is_file():
        audit = read_json(audit_path)
        if (audit.get('seed') != CONFIG['seed']
                or audit.get('near_threshold') != NEAR_THRESHOLD
                or audit.get('dataset_archive_sha256') != DATASET_SHA256):
            raise ValueError(
                'Dataset atau pengaturan audit berbeda dari manifest tersimpan. '
                'Gunakan DATA_VERSION dan RUN_PREFIX baru.'
            )
        saved_rows = load_manifest(MANIFEST)
        if manifest_fingerprint(saved_rows) != audit.get('fingerprint'):
            raise ValueError('Manifest berubah dari hasil audit tersimpan.')
        verify_data_files(DATA, saved_rows)
        print('Manifest tersimpan digunakan kembali.')
        return audit
    if not RUN_PREPARE:
        raise ValueError('Manifest belum ada. Aktifkan RUN_PREPARE lalu jalankan audit.')
    print('Audit data dimulai; puluhan ribu gambar dapat memerlukan beberapa menit.', flush=True)
    audit = prepare_data(DATA, PROCESSED, seed=CONFIG['seed'], near_threshold=NEAR_THRESHOLD)
    audit['dataset_archive_sha256'] = DATASET_SHA256
    write_json(audit_path, audit)
    return audit

def train_stage(method, init_checkpoint=None):
    if not RUN_TRAINING:
        print(f'{method.upper()}: training dinonaktifkan.')
        return None
    run = RUNS[method]
    metadata_path = run / 'run.json'
    if metadata_path.is_file():
        metadata = read_json(metadata_path)
        if metadata.get('status') != 'complete':
            raise RuntimeError(f'{run.name} terputus. Gunakan RUN_PREFIX baru untuk mengulang.')
        if (metadata.get('method') != method or metadata.get('config') != CONFIG
                or metadata.get('manifest_fingerprint') != manifest_fingerprint(load_manifest(MANIFEST))):
            raise RuntimeError('Run memakai konfigurasi/split berbeda. Gunakan RUN_PREFIX baru.')
        checkpoint = run / ('b0.json' if method == 'b0' else 'best.pt')
        if not checkpoint.is_file():
            raise RuntimeError('Checkpoint hilang. Gunakan RUN_PREFIX baru.')
        if method in {'m1', 'b1_control'}:
            if not init_checkpoint or archive_hash(init_checkpoint) != metadata.get('source_checkpoint_sha256'):
                raise RuntimeError('Checkpoint B1 awal berubah. Gunakan RUN_PREFIX baru.')
        # Verify the saved checkpoint, manifest, and data before reusing the result.
        evaluate(run, DATA, MANIFEST, device=DEVICE)
        print(f'{method.upper()}: hasil selesai di Drive digunakan kembali.')
        return metadata
    return train(method, DATA, MANIFEST, run, CONFIG, DEVICE, init_checkpoint)


## 2. Jalankan audit dan lihat distribusi usia
Tinjau `invalid_files.csv`, `near_duplicates.csv`, dan `duplicate_groups.csv`
pada folder processed sebelum eksperimen penelitian. Audit tidak menjamin
pemisahan orang yang sama antar-split karena UTKFace tidak menyediakan ID lengkap.

In [ ]:
audit = prepare_notebook_data()
display(audit)
rows = load_manifest(MANIFEST)
print('Jumlah citra:', len(rows))
print('Fingerprint:', manifest_fingerprint(rows))
print('Split:', {name: sum(row['split'] == name for row in rows) for name in SPLITS})

plt = _pyplot()
fig, ax = plt.subplots(figsize=(8, 4))
for split in SPLITS:
    ages = [row['age'] for row in rows if row['split'] == split]
    ax.hist(ages, bins=list(range(0, 121, 5)), alpha=.45, label=split)
ax.set(xlabel='Label usia (tahun)', ylabel='Jumlah citra', title='Distribusi label UTKFace')
ax.legend()
fig.tight_layout()
fig.savefig(PROCESSED / 'age_distribution.png', dpi=170)
display(fig)
plt.close(fig)


## 3. Baseline B0 median usia
Median dihitung hanya dari label data latih; citra tidak digunakan.
Checkpoint dan history disimpan langsung ke Drive.

In [ ]:
display(train_stage('b0'))


## 4. Baseline B1 backbone beku
Fitur ImageNet dipertahankan. Classifier dan head regresi dilatih; backbone dan BatchNorm beku.
Checkpoint dan history disimpan langsung ke Drive.

In [ ]:
display(train_stage('b1'))


## 5. Metode M1 fine tuning
Mulai dari checkpoint B1 terbaik. Tiga modul terakhir backbone dan head dilatih.
Checkpoint dan history disimpan langsung ke Drive.

In [ ]:
display(train_stage('m1', init_checkpoint=RUNS['b1'] / 'best.pt'))


## 6. B1 control tambahan pelatihan head
Mulai dari checkpoint B1 yang sama dengan M1, lalu lanjutkan head dengan backbone beku. Batas epoch sama; early stopping dapat memberi jumlah epoch aktual berbeda.
Checkpoint dan history disimpan langsung ke Drive.

In [ ]:
display(train_stage('b1_control', init_checkpoint=RUNS['b1'] / 'best.pt'))


## 7. Perbandingan validasi
Pilih model dan konfigurasi dari hasil validasi. Target penurunan MAE 20%
terhadap B0 merupakan sasaran proposal, bukan angka hasil yang diasumsikan.
Tabel seluruh metode dan status kontrol fine tuning tersimpan ke Drive.

In [ ]:
if RUN_COMPARE:
    comparison = compare_runs(list(RUNS.values()), DATA, MANIFEST, COMPARISON,
                              split='validation', device=DEVICE)
    display(comparison)
    display(DisplayImage(filename=str(COMPARISON / 'comparison_validation.png')))
else:
    print('Perbandingan validasi dinonaktifkan.')

for method, run in RUNS.items():
    scores = run / 'evaluation_validation' / 'metrics.json'
    if scores.is_file():
        print(method.upper())
        display(read_json(scores))


## 8. Grafik dan kasus gagal model terpilih
`SELECTED_METHOD` ditentukan pada sel konfigurasi. Grafik memakai prediksi
mentah. Tabel kasus gagal menyimpan path dan galat; alasan kegagalan perlu
kamu analisis dari citra, bukan dihasilkan otomatis sebagai fakta.

In [ ]:
evaluation_dir = RUNS[SELECTED_METHOD] / 'evaluation_validation'
for filename in ('learning_curve.png', 'prediction_vs_label.png',
                 'residual_histogram.png', 'age_group_mae.png'):
    path = evaluation_dir / filename
    if path.is_file():
        print(filename)
        display(DisplayImage(filename=str(path)))
worst = evaluation_dir / 'worst_examples.csv'
if worst.is_file():
    display(read_csv(worst)[:10])
else:
    print('Jalankan training dan evaluasi validasi dahulu.')


## 9. Evaluasi akhir test set
Setelah model dan konfigurasi dipilih, ubah `RUN_FINAL_TEST = True` pada
sel konfigurasi, set flag training/compare ke False, jalankan ulang konfigurasi
dan sel ini. Jangan memilih model atau menyetel parameter berdasarkan skor test.

In [ ]:
if RUN_FINAL_TEST:
    display(compare_runs(list(RUNS.values()), DATA, MANIFEST, COMPARISON,
                         split='test', device=DEVICE, final_test=True))
    display(DisplayImage(filename=str(COMPARISON / 'comparison_test.png')))
else:
    print('Test set belum dibuka. RUN_FINAL_TEST = False.')


## 10. Unggah foto dan prediksi di notebook
Setelah checkpoint penelitian tersedia, ubah `RUN_PHOTO_UPLOAD = True`.
Pilih tepat satu JPG/PNG wajah yang sudah dipotong. Foto diproses pada runtime;
sel tidak menyalinnya ke Drive. Foto dan output dapat ikut tersimpan bersama
notebook jika output disimpan. Gunakan foto dengan izin.

In [ ]:
RUN_PHOTO_UPLOAD = False
if RUN_PHOTO_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Pilih tepat satu foto wajah JPG/PNG.')
    name, content = next(iter(uploaded.items()))
    image = decode_image(content)
    display(image)
    checkpoint = RUNS[SELECTED_METHOD] / ('b0.json' if SELECTED_METHOD == 'b0' else 'best.pt')
    predictor = Predictor(checkpoint, device=DEVICE)
    display(predictor.predict(image))
else:
    print('Aktifkan RUN_PHOTO_UPLOAD setelah model terlatih tersedia.')


## 11. Logbook kosong dan lokasi hasil
Sel berikut membuat 13 baris kosong bila file belum ada. Logbook yang sudah
kamu isi tidak ditimpa. Catat pekerjaan nyata, tanggal, kendala, dan bukti sendiri.

- `processed/<DATA_VERSION>/`: audit, manifest, distribusi usia.
- `runs/<RUN_PREFIX>_<metode>/`: checkpoint, konfigurasi, metadata, history,
  CSV prediksi, metrik, dan grafik.
- `runs/<RUN_PREFIX>_comparison/`: perbandingan validasi dan test bila diaktifkan.

Disk runtime sementara. Saat sesi baru, jalankan setup, ekstraksi dataset,
definisi fungsi, konfigurasi, dan audit lagi. Manifest yang cocok dan tahap
selesai digunakan kembali; training yang terputus perlu nama run baru.

In [ ]:
logbook_path = DRIVE_PROJECT / 'logbook_kosong.csv'
if not logbook_path.exists():
    logbook_fields = ['Nama', 'NIM', 'Kelompok', 'Minggu', 'Tanggal', 'Target',
                      'Kegiatan nyata', 'Keputusan atau hasil belajar', 'Bukti',
                      'Kendala', 'Rencana berikutnya']
    write_csv(logbook_path, [{field: '' for field in logbook_fields} for _ in range(13)],
              logbook_fields)
print('Logbook:', logbook_path)
print('Audit dan manifest:', PROCESSED)
print('Checkpoint, metrik, grafik:', DRIVE_PROJECT / 'runs')


## Referensi dan batas verifikasi
- Dataset dan ketentuan riset: [UTKFace AICIP](https://susanqq.github.io/UTKFace/).
- Salinan arsip cropped: [UTKFace Aligned & Cropped di Kaggle](https://www.kaggle.com/datasets/moritzm00/utkface-cropped).
- Model: Howard dkk., [Searching for MobileNetV3](https://openaccess.thecvf.com/content_ICCV_2019/html/Howard_Searching_for_MobileNetV3_ICCV_2019_paper.html).
- Praproses/bobot: [torchvision MobileNetV3 Small](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.mobilenet_v3_small.html).
- Lingkungan: [FAQ resmi Colab](https://research.google.com/colaboratory/faq.html).

Alur notebook diuji secara lokal dengan ZIP berisi gambar sintetis, CPU,
dan simulasi lokasi Drive tanpa folder repo. Simulasi tidak mengukur akurasi
UTKFace atau membuktikan mount Drive/GPU pada akun Colab. Jalankan penelitian
sebenarnya pada dataset UTKFace; catat versi dan perangkat aktual dari `run.json`.